# SOTA Championship Cross-Modal Person Retrieval: Kaggle Dual Tesla T4 (30 Epoch Championship Run)
### WACV 2027 UPAR Challenge & MEVID Benchmark | Target: mADM >= 0.7000
---
This notebook executes the **30-epoch full championship training and submission pipeline** for cross-modal person retrieval.
Features:
- **Authentic Pretrained Foundation Backbone:** EVA-02-CLIP (No fallbacks permitted)
- **Anatomical 3-Strip Partitioning:** Head (25%), Torso (37.5%), Legs (37.5%) with learnable GeM Pooling
- **SOTA Loss Suite:** Soft Multi-Instance SDM + MLM Cross-Attention + Asymmetric Loss (ASL)
- **Dual-GPU Distributed Retrieval:** Full AMP autocast + linear chunked dual-GPU validation (<30s runtime, ~13.0 GB VRAM per GPU)
- **High-Throughput VRAM Maximization:** 96 samples/GPU (~13.5-14.2 GB VRAM per GPU)
- **100% Loss-Free Checkpointing:** Seamless resumption across Kaggle sessions
- **Automated Submission Packaging:** Exports FP16 weights directly to air-gapped `submission.zip`
- **Isolated Sandbox Structural Audit:** 10-point structural integrity verification inside a clean container sandbox (no forward pass)


In [ ]:
# ==============================================================================
# CELL 1: HARDWARE & SYSTEM TELEMETRY CHECK
# ==============================================================================
import os
import sys
import psutil
import torch

print("=" * 80)
print("[HARDWARE] Inspecting Dual Tesla T4 Accelerators & Host Environment...")
print("=" * 80)

cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f"CUDA Available: {cuda_available} | Device Count: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  -> GPU {i}: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")

vmem = psutil.virtual_memory()
print(f"Host CPU Cores: {os.cpu_count()} | Total RAM: {vmem.total / (1024**3):.2f} GB | Available: {vmem.available / (1024**3):.2f} GB")
print("=" * 80)

# Install required SOTA vision-language dependencies
print("[DEPS] Installing OpenCLIP and Timm dependencies...")
import subprocess
subprocess.run(["pip", "install", "-q", "open-clip-torch", "timm"], check=True)
print("[DEPS] Dependencies installed successfully!")


In [ ]:
# ==============================================================================
# CELL 2: CLONE OR PULL PRIVATE REPOSITORY VIA GITHUB_TOKEN
# ==============================================================================
import os
import sys
import subprocess

REPO_NAME = "upar_challenge_26_rws27"
REPO_DIR = os.path.join("/kaggle/working", REPO_NAME)

# Retrieve Kaggle Secret GITHUB_TOKEN
token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("GITHUB_TOKEN")
    print("[AUTH] Successfully retrieved GITHUB_TOKEN from Kaggle Secrets!")
except Exception as e:
    print(f"[AUTH] Note: Kaggle Secrets not accessible ({e}). Attempting public clone...")

if token:
    repo_url = f"https://{token}@github.com/Aditya-Lingam-9000/{REPO_NAME}.git"
else:
    repo_url = f"https://github.com/Aditya-Lingam-9000/{REPO_NAME}.git"

if not os.path.isdir(REPO_DIR):
    print(f"[GIT] Cloning repository into {REPO_DIR}...")
    res = subprocess.run(["git", "clone", repo_url, REPO_DIR], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[GIT ERROR] Clone failed:\n{res.stderr}")
        raise RuntimeError("Failed to clone private repository. Please verify GITHUB_TOKEN.")
    print(f"[GIT] Successfully cloned repository!")
else:
    print(f"[GIT] Repository already exists at {REPO_DIR}. Fetching latest commits from main...")
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    print(res.stdout)

# Add repository root to Python path and change working directory
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"[SYSTEM] Working directory set to: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 3: MASTER CHAMPIONSHIP CONFIGURATION PANEL
# ==============================================================================
import os
import sys
from sota_pipeline.config import SOTAConfig

# Modify any training hyperparameter here before running training.
EPOCHS = 30                  # 30 Epochs: Full Championship Run (Target: mADM >= 0.7000)
BATCH_SIZE_PER_GPU = 96      # 96 per GPU * 2 GPUs = Global: 192 (Optimized to utilize ~13.5-14.2 GB VRAM per GPU)
ACCUMULATION_STEPS = 2       # Gradient accumulation: Effective global batch size = 384
BACKBONE_LR = 5e-5           # Conservative learning rate for pretrained visual backbone
ADAPTER_LR = 3.5e-4          # Higher learning rate for cross-modal heads and adapters
MIN_LR = 1e-6                # Minimum learning rate floor for Cosine Annealing
LOG_INTERVAL = 25            # Step interval for step-wise formatted console logs
EVAL_EVERY = 1               # Run full validation retrieval after EVERY epoch
VAL_QUERIES = 500            # Number of validation queries for retrieval benchmarking
EVAL_BATCH_SIZE = 384        # Validation inference batch size (Dual-GPU distributed, ~13.0 GB VRAM, finishes in <30s)
MAX_SESSION_HOURS = 10.75    # Auto-save & graceful exit before Kaggle 12-hour session kill
CHECKPOINT_DIR = "checkpoints_sota"
OUTPUT_DIR = "sota_outputs"

# Auto-detect UPAR dataset root via dynamic deep scanner
DATASET_ROOT = None
try:
    DATASET_ROOT = SOTAConfig.resolve_dataset_root()
except FileNotFoundError as err:
    print(f"\n[DATASET RESOLUTION ERROR]\n{err}\n")
    raise err

print("=" * 80)
print(f"[CONFIG] Training Configuration:")
print(f"  - Epochs: {EPOCHS} (Full 30-Epoch Championship Run, Multi-Session Resumable)")
print(f"  - Batch Size: {BATCH_SIZE_PER_GPU}/GPU (Global: {BATCH_SIZE_PER_GPU * 2}, Effective: {BATCH_SIZE_PER_GPU * 2 * ACCUMULATION_STEPS})")
print(f"  - Backbone LR: {BACKBONE_LR} | Adapters LR: {ADAPTER_LR} | Min LR: {MIN_LR}")
print(f"  - Eval Interval: Every {EVAL_EVERY} epoch(s) | Val Queries: {VAL_QUERIES}")
print(f"  - Eval Batch Size: {EVAL_BATCH_SIZE} (Distributed across 2 GPUs, ~13 GB VRAM)")
print(f"  - Dataset Root: {DATASET_ROOT}")
print(f"  - Checkpoint Dir: {os.path.abspath(CHECKPOINT_DIR)}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 4: HUGGINGFACE ACCELERATE DUAL T4 MULTI-GPU CONFIGURATION
# ==============================================================================
import os
import yaml

accelerate_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "MULTI_GPU",
    "num_machines": 1,
    "num_processes": 2,
    "gpu_ids": "all",
    "mixed_precision": "fp16",
    "main_process_port": 29500,
    "rdzv_backend": "static",
    "same_network": True,
}

config_path = "accelerate_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(accelerate_config, f, default_flow_style=False)

print(f"[ACCELERATE] Generated multi-GPU configuration at: {os.path.abspath(config_path)}")


In [ ]:
# ==============================================================================
# CELL 5: LAUNCH SOTA CHAMPIONSHIP DISTRIBUTED TRAINING
# ==============================================================================
# Executes directly from the cloned repository (sota_pipeline.train).
# Features:
# 1. Baseline Pretrained Zero-Shot Validation (Epoch 0)
# 2. Lightning-fast RAM Preloading (2.5s from HDF5 container, zero disk bottlenecks)
# 3. Clean step-wise logs (NO tqdm bar clutter) with loss components, ETAs, speed, and VRAM
# 4. Full validation retrieval after EVERY epoch (mADM, CMC Rank-1/5/10, mAP)
# 5. Strictly saves ONLY best_model.pt and last_checkpoint.pt
# 6. Automatic resumption from last_checkpoint.pt if restarting session
import os
import sys
import shutil
import subprocess

# Auto-detect launcher: accelerate launch or torchrun fallback
if shutil.which("accelerate"):
    launcher = ["accelerate", "launch", "--config_file", "accelerate_config.yaml"]
else:
    launcher = ["torchrun", "--nproc_per_node=2"]

cmd = launcher + [
    "-m", "sota_pipeline.train",
    f"--epochs={EPOCHS}",
    f"--batch_size={BATCH_SIZE_PER_GPU}",
    f"--accumulation_steps={ACCUMULATION_STEPS}",
    f"--backbone_lr={BACKBONE_LR}",
    f"--adapter_lr={ADAPTER_LR}",
    f"--min_lr={MIN_LR}",
    f"--log_interval={LOG_INTERVAL}",
    f"--eval_every={EVAL_EVERY}",
    f"--val_queries={VAL_QUERIES}",
    f"--eval_batch_size={EVAL_BATCH_SIZE}",
    f"--max_session_hours={MAX_SESSION_HOURS}",
    f"--checkpoint_dir={CHECKPOINT_DIR}",
    f"--output_dir={OUTPUT_DIR}",
    f"--dataset_root={DATASET_ROOT}",
    "--run_baseline",
]

print("[EXEC] Executing command:", " ".join(cmd))
print("=" * 80)
subprocess.run(cmd, check=True)


In [ ]:
# ==============================================================================
# CELL 6: POST-TRAINING INTEGRITY AUDIT & VERIFICATION
# ==============================================================================
# Executes directly from the cloned repository (sota_pipeline.verify).
# Verifies:
# 1. Existence and sizes of strictly best_model.pt and last_checkpoint.pt
# 2. Confirms zero intermediate checkpoint bloat
# 3. Verifies all 7 multi-session state components in each checkpoint
# 4. Validates model weight loading and executes a forward test pass
import os
import sys
import subprocess
import torch

audit_cmd = [
    sys.executable, "-m", "sota_pipeline.verify",
    f"--checkpoint_dir={CHECKPOINT_DIR}",
    "--device=cuda:0" if torch.cuda.is_available() else "--device=cpu",
]
print("[AUDIT] Running:", " ".join(audit_cmd))
subprocess.run(audit_cmd, check=True)


In [ ]:
# ==============================================================================
# CELL 7: AUTOMATED SOTA SUBMISSION ZIP PACKAGING
# ==============================================================================
# Packages the best trained checkpoint directly into a self-contained, air-gapped
# CodaBench submission ZIP file (/kaggle/working/submission.zip).
# Features:
# 1. Extracts pure FP16 weights from best_model.pt (zero unneeded optimizer/grad state)
# 2. Embeds authentic BPE vocabulary and attribute priors
# 3. Bundles self-contained entrypoint run.py and metadata.yaml
# 4. Bundles vendored timm library (zero internet dependency)
# 5. Strictly enforces < 500 MB total archive size
import os
import sys
import shutil
import zipfile
import torch

print("=" * 80)
print("[PACKAGING] Generating CodaBench Submission ZIP Archive...")
print("=" * 80)

# 1. Locate best checkpoint
ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
if not os.path.isfile(ckpt_path):
    ckpt_path = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")
if not os.path.isfile(ckpt_path):
    raise FileNotFoundError(f"No checkpoint found in {CHECKPOINT_DIR}! Run training first.")

print(f"[PACKAGING] Using source checkpoint: {ckpt_path}")

# 2. Setup build directory
BUILD_DIR = os.path.join("/kaggle/working", "submission_build")
if os.path.exists(BUILD_DIR):
    shutil.rmtree(BUILD_DIR)
os.makedirs(os.path.join(BUILD_DIR, "assets"), exist_ok=True)

# 3. Convert and extract FP16 model weights
print("[PACKAGING] Extracting FP16 model weights...")
checkpoint_data = torch.load(ckpt_path, map_location="cpu", weights_only=False)
raw_state_dict = checkpoint_data.get("model_state_dict", checkpoint_data)

fp16_state_dict = {}
for key, val in raw_state_dict.items():
    # Strip module. prefix if saved via DistributedDataParallel
    clean_key = key.replace("module.", "")
    if isinstance(val, torch.Tensor) and val.is_floating_point():
        fp16_state_dict[clean_key] = val.half()
    else:
        fp16_state_dict[clean_key] = val

weights_out_path = os.path.join(BUILD_DIR, "assets", "model_sota_fp16.pt")
torch.save(fp16_state_dict, weights_out_path)
weights_size_mb = os.path.getsize(weights_out_path) / (1024 * 1024)
print(f"[PACKAGING] Extracted {len(fp16_state_dict)} FP16 tensors ({weights_size_mb:.2f} MB) -> {weights_out_path}")

# 4. Copy runtime code, metadata, assets, and vendored dependencies
SRC_SUBMISSION_DIR = os.path.join(os.getcwd(), "sota_pipeline/submission_template")
if not os.path.isdir(SRC_SUBMISSION_DIR):
    SRC_SUBMISSION_DIR = os.path.join(os.getcwd(), "submissions/SOTA_Full_Championship_Submission")

shutil.copy(os.path.join(SRC_SUBMISSION_DIR, "metadata.yaml"), os.path.join(BUILD_DIR, "metadata.yaml"))
shutil.copy(os.path.join(SRC_SUBMISSION_DIR, "run.py"), os.path.join(BUILD_DIR, "run.py"))
shutil.copy(os.path.join(SRC_SUBMISSION_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz"), os.path.join(BUILD_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz"))
shutil.copy(os.path.join(SRC_SUBMISSION_DIR, "assets", "attr_priors.npy"), os.path.join(BUILD_DIR, "assets", "attr_priors.npy"))

vendor_src = os.path.join(SRC_SUBMISSION_DIR, "vendor")
if os.path.exists(vendor_src):
    shutil.copytree(vendor_src, os.path.join(BUILD_DIR, "vendor"))

# 5. Create final ZIP package
ZIP_DEST = os.path.join("/kaggle/working", "submission.zip")
if os.path.exists(ZIP_DEST):
    os.remove(ZIP_DEST)

print(f"[PACKAGING] Compressing assets into: {ZIP_DEST}...")
with zipfile.ZipFile(ZIP_DEST, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(BUILD_DIR):
        for file in files:
            abs_p = os.path.join(root, file)
            rel_p = os.path.relpath(abs_p, BUILD_DIR)
            zf.write(abs_p, rel_p)

final_zip_mb = os.path.getsize(ZIP_DEST) / (1024 * 1024)
print("=" * 80)
print(f"[PACKAGING COMPLETE] Successfully built: {ZIP_DEST}")
print(f"  - Package Archive Size: {final_zip_mb:.2f} MB")
print(f"  - Weight State Dict: {weights_size_mb:.2f} MB (FP16)")
print(f"  - Ready for immediate CodaBench evaluation!")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 8: ISOLATED SANDBOX CONTAINER STRUCTURAL AUDIT & VALIDATION
# ==============================================================================
# Validates the submission archive in a strict, isolated container sandbox
# WITHOUT executing model forward passes (verifies packaging, air-gapping,
# weight dictionary structure, and clean container hygiene).
import os
import sys
import ast
import shutil
import zipfile
import tempfile
import numpy as np
import torch

ZIP_PATH = os.path.join("/kaggle/working", "submission.zip")
if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Submission ZIP not found at {ZIP_PATH}! Run Cell 7 first.")

SANDBOX_DIR = tempfile.mkdtemp(prefix="codabench_sandbox_audit_")
try:
    print("=" * 80)
    print(f"[SANDBOX AUDIT] Extracting {ZIP_PATH} into clean sandbox: {SANDBOX_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(SANDBOX_DIR)

    audit_checks = []

    # 1. run.py existence & non-empty
    r_path = os.path.join(SANDBOX_DIR, "run.py")
    c1 = os.path.isfile(r_path) and os.path.getsize(r_path) > 10000
    audit_checks.append(("run.py existence and integrity", c1, f"{os.path.getsize(r_path):,} bytes"))

    # 2. metadata.yaml existence
    m_path = os.path.join(SANDBOX_DIR, "metadata.yaml")
    c2 = os.path.isfile(m_path)
    audit_checks.append(("metadata.yaml descriptor presence", c2, "Found"))

    # 3. Model weights existence & size
    w_path = os.path.join(SANDBOX_DIR, "assets", "model_sota_fp16.pt")
    c3 = os.path.isfile(w_path) and os.path.getsize(w_path) > 250 * 1024 * 1024
    w_mb = os.path.getsize(w_path) / (1024 * 1024) if os.path.isfile(w_path) else 0
    audit_checks.append(("model_sota_fp16.pt (> 250 MB)", c3, f"{w_mb:.2f} MB"))

    # 4. Standalone BPE vocabulary existence
    b_path = os.path.join(SANDBOX_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
    c4 = os.path.isfile(b_path) and os.path.getsize(b_path) > 1024 * 1024
    audit_checks.append(("bpe_simple_vocab_16e6.txt.gz (> 1 MB)", c4, "Found"))

    # 5. Attribute priors validity
    p_path = os.path.join(SANDBOX_DIR, "assets", "attr_priors.npy")
    priors = np.load(p_path) if os.path.isfile(p_path) else None
    c5 = os.path.isfile(p_path) and priors is not None and priors.shape == (40,)
    audit_checks.append(("attr_priors.npy (shape: [40])", c5, f"{priors.shape if priors is not None else None}"))

    # 6. Vendored timm module for zero-internet execution
    v_path = os.path.join(SANDBOX_DIR, "vendor", "timm")
    c6 = os.path.isdir(v_path) and os.path.isfile(os.path.join(v_path, "__init__.py"))
    audit_checks.append(("vendor/timm air-gapped module", c6, "Directory verified"))

    # 7. AST syntax parse & CodaBench entrypoints check
    with open(r_path, "r", encoding="utf-8") as f:
        code_text = f.read()
    parsed_tree = ast.parse(code_text)
    func_names = [n.name for n in ast.walk(parsed_tree) if isinstance(n, ast.FunctionDef)]
    c7 = all(fn in func_names for fn in ["load_model", "rank_gallery"])
    audit_checks.append(("CodaBench entrypoints (load_model, rank_gallery)", c7, "Both confirmed"))

    # 8. Air-gapped code hygiene (no hardcoded absolute machine paths)
    c8 = ("/kaggle/" not in code_text) and ("/Users/" not in code_text) and ("/home/" not in code_text)
    audit_checks.append(("Air-gapped code hygiene (no hardcoded host paths)", c8, "Clean"))

    # 9. Structural weight dictionary inspection (without forward pass)
    sd = torch.load(w_path, map_location="cpu", weights_only=False)
    v_keys = sum(1 for k in sd if k.startswith("visual_stream."))
    t_keys = sum(1 for k in sd if k.startswith("text_stream."))
    all_fp16 = all(v.dtype == torch.float16 for v in sd.values() if isinstance(v, torch.Tensor) and v.is_floating_point())
    has_nan = any(torch.isnan(v).any().item() for v in sd.values() if isinstance(v, torch.Tensor))
    has_inf = any(torch.isinf(v).any().item() for v in sd.values() if isinstance(v, torch.Tensor))
    c9 = (v_keys == 280) and (t_keys == 155) and all_fp16 and (not has_nan) and (not has_inf)
    audit_checks.append(("FP16 weight structure (280 visual, 155 text, 0 NaNs/Infs)", c9, f"{v_keys} visual, {t_keys} text"))

    # 10. Archive size compliance
    zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    c10 = zip_mb < 500.0
    audit_checks.append(("Archive size compliance (< 500 MB)", c10, f"{zip_mb:.2f} MB"))

    print("=" * 80)
    print("ISOLATED SANDBOX CONTAINER STRUCTURAL AUDIT SCORECARD")
    print("=" * 80)
    all_passed = True
    for name, passed, detail in audit_checks:
        status = "[PASS]" if passed else "[FAIL]"
        print(f"  {status:7s} | {name:<50s} | {detail}")
        if not passed:
            all_passed = False
    print("=" * 80)
    if all_passed:
        print("[AUDIT VERDICT] ALL 10/10 CHECKS PASSED WITH FLYING COLORS!")
        print("  -> Package is 100% self-contained, air-gapped, and competition-ready.")
    else:
        print("[AUDIT VERDICT] AUDIT FAILED! Review failing checks above.")
        raise RuntimeError("Structural audit failed! Do not submit this archive.")
    print("=" * 80)
finally:
    if os.path.exists(SANDBOX_DIR):
        shutil.rmtree(SANDBOX_DIR)
        print(f"[CLEANUP] Successfully removed sandbox directory: {SANDBOX_DIR}")
